# Create Smithsonian Artist Research Fellowship Awards

Creates OpenAlex award rows from the Smithsonian Office of Academic Appointments and Internships official SARF recipient list.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/smithsonian_sarf_to_s3.py` to fetch the official recipient/program pages, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** `https://fellowships.si.edu/SARFawards` plus the SARF program page at `https://fellowships.si.edu/SARF`.

**S3 location:** `s3a://openalex-ingest/awards/smithsonian_sarf/smithsonian_sarf_awards.parquet`

**Local full-source validation on 2026-05-27:** 101 fellowship recipient rows, years 2017-2024, 100% recipient/year/host-unit/landing-url coverage, 99.0% project-title coverage.

**OpenAlex funder mapping:** `F7230414656` Office of Fellowships, Smithsonian Institution. This exact fellowship-office row is more specific than the main Smithsonian Institution funder for this SARF recipient source.

**Amount/currency:** NULL with section 6.7 waiver. The SARF program page publishes the current stipend policy (`$5,000 per month`), but exact recipient tenure dates and amounts are specified in individual award letters and are not published on the recipient list. The raw parquet preserves `current_program_stipend_raw` and `amount_decision` for audit.

**Mapping summary:** one row per official SARF recipient paragraph. `lead_investigator` is the fellowship recipient; `lead_investigator.affiliation.name` is the Smithsonian host unit(s) named by the source. Smithsonian advisors are preserved in raw staging only, not mapped as investigators.

## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.smithsonian_sarf_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/smithsonian_sarf/smithsonian_sarf_awards.parquet`;

In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-27 found 101 rows.
SELECT COUNT(*) AS total_smithsonian_sarf_rows
FROM openalex.awards.smithsonian_sarf_raw;

In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.smithsonian_sarf_raw;

In [ ]:
%sql
-- Sample raw SARF data.
SELECT
    funder_award_id,
    source_year,
    recipient_name,
    project_title,
    host_unit,
    smithsonian_advisors,
    current_program_stipend_raw,
    landing_page_url
FROM openalex.awards.smithsonian_sarf_raw
LIMIT 10;

## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'smithsonian_sarf_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|stipend|allowance|award|grant';

In [ ]:
%sql
-- Confirm coverage and source year shape before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(recipient_name) AS rows_with_recipient,
    COUNT(project_title) AS rows_with_project_title,
    COUNT(host_unit) AS rows_with_host_unit,
    COUNT(start_date) AS rows_with_start_date,
    COUNT(end_date) AS rows_with_end_date,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(project_title), COUNT(*)) * 100.0, 1) AS pct_project_title,
    ROUND(try_divide(COUNT(host_unit), COUNT(*)) * 100.0, 1) AS pct_host_unit,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    MIN(TRY_CAST(source_year AS INT)) AS min_year,
    MAX(TRY_CAST(source_year AS INT)) AS max_year
FROM openalex.awards.smithsonian_sarf_raw;

In [ ]:
%sql
-- Native-key inspection. funder_award_id must be unique.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.smithsonian_sarf_raw;

In [ ]:
%sql
-- Year distribution from the official recipient headings.
SELECT source_year, COUNT(*) AS recipients
FROM openalex.awards.smithsonian_sarf_raw
GROUP BY source_year
ORDER BY TRY_CAST(source_year AS INT) DESC;

In [ ]:
%sql
-- Host-unit distribution.
SELECT host_unit, COUNT(*) AS recipients
FROM openalex.awards.smithsonian_sarf_raw
GROUP BY host_unit
ORDER BY recipients DESC, host_unit
LIMIT 30;

In [ ]:
%sql
-- Amount/currency decision audit fields.
SELECT
    current_program_stipend_raw,
    amount_decision,
    COUNT(*) AS rows
FROM openalex.awards.smithsonian_sarf_raw
GROUP BY current_program_stipend_raw, amount_decision;

## Step 1.6: Funder Row (Path B — inline constants)

`F7230414656` is a **non-F4320\*** funder, so `openalex.common.funder` (which mirrors only F4320\* Crossref-registered funders) does **not** contain a row for it. Per `plans/awards/how-to-add-a-funder-v2.md` §1.6 Path B, the Step 2 transform must source the funder row from inline constants populated from `https://api.openalex.org/funders/F7230414656`, not from the dim.

The informational SELECT below confirms 0 rows in the dim (expected for Path B). Do not gate on it.

In [ ]:
%sql
-- Path B funder: 0 rows in openalex.common.funder is EXPECTED for F7230414656.
-- Informational only; do not assert.
SELECT COUNT(*) AS dim_row_count
FROM openalex.common.funder
WHERE funder_id = 7230414656;

In [ ]:
%sql
-- Canonical funder values for Step 2 (sourced from https://api.openalex.org/funders/F7230414656).
-- These are the inline constants the Step 2 CTE uses.
SELECT
    7230414656 AS funder_id,
    'Office of Fellowships, Smithsonian Institution' AS display_name,
    CAST(NULL AS STRING) AS ror_id,
    '10.13039/100023983' AS doi;

## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.smithsonian_sarf_awards
USING delta
AS
WITH
smithsonian_funder AS (
    -- INLINED canonical funder row (Path B: non-F4320*, not in openalex.common.funder).
    -- Values from https://api.openalex.org/funders/F7230414656 per how-to-add-a-funder-v2.md §1.6.
    SELECT
        7230414656 AS funder_id,
        'Office of Fellowships, Smithsonian Institution' AS display_name,
        CAST(NULL AS STRING) AS ror_id,
        '10.13039/100023983' AS doi
),
raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(source_year AS INT) AS parsed_year,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date
    FROM openalex.awards.smithsonian_sarf_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 AS id,

        TRIM(g.display_name) AS display_name,
        NULLIF(TRIM(g.description), '') AS description,

        f.funder_id,
        g.native_award_id AS funder_award_id,

        CAST(NULL AS DOUBLE) AS amount,
        CAST(NULL AS STRING) AS currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,

        'fellowship' AS funding_type,
        COALESCE(NULLIF(TRIM(g.funder_scheme), ''), 'Smithsonian Artist Research Fellowship') AS funder_scheme,

        'smithsonian_sarf' AS provenance,

        g.parsed_start_date AS start_date,
        g.parsed_end_date AS end_date,
        g.parsed_year AS start_year,
        g.parsed_year AS end_year,

        struct(
            NULLIF(TRIM(g.given_name), '') AS given_name,
            NULLIF(TRIM(g.family_name), '') AS family_name,
            CAST(NULL AS STRING) AS orcid,
            g.parsed_start_date AS role_start,
            struct(
                NULLIF(TRIM(g.host_unit), '') AS name,
                'US' AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        ) AS lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,

        g.landing_page_url AS landing_page_url,
        CAST(NULL AS STRING) AS doi,

        concat(
            'https://api.openalex.org/works?filter=awards.id:G',
            abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000
        ) AS works_api_url,

        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM raw_prepared g
    CROSS JOIN smithsonian_funder f
)
SELECT * FROM awards_transformed;

## Step 3: Insert Into `openalex_awards_raw` With Priority 132

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'smithsonian_sarf' AND priority = 132;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    132 AS priority
FROM openalex.awards.smithsonian_sarf_awards;

## Step 6: Verification Queries

In [ ]:
%sql
SELECT COUNT(*) AS total_smithsonian_sarf_awards
FROM openalex.awards.smithsonian_sarf_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.smithsonian_sarf_awards;

In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    start_year,
    lead_investigator.given_name AS recipient_given,
    lead_investigator.family_name AS recipient_family,
    lead_investigator.affiliation.name AS host_unit,
    landing_page_url
FROM openalex.awards.smithsonian_sarf_awards
LIMIT 10;

In [ ]:
%sql
-- ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.smithsonian_sarf_awards;

In [ ]:
%sql
-- Funder distribution. Should be only F7230414656.
SELECT funder.display_name, funder_id, COUNT(*) AS cnt
FROM openalex.awards.smithsonian_sarf_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;

In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    COUNT(landing_page_url) AS has_url,
    COUNT(lead_investigator.given_name) AS has_recipient_given,
    COUNT(lead_investigator.family_name) AS has_recipient_family,
    COUNT(lead_investigator.affiliation.name) AS has_host_unit,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(lead_investigator.family_name), COUNT(*)) * 100.0, 1) AS pct_recipient_family,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name), COUNT(*)) * 100.0, 1) AS pct_host_unit
FROM openalex.awards.smithsonian_sarf_awards;

In [ ]:
%sql
-- Section 6.7 amount/currency check (waived for this fellowship source).
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies
FROM openalex.awards.smithsonian_sarf_awards;

In [ ]:
%sql
-- Year distribution.
SELECT start_year, COUNT(*) AS recipients
FROM openalex.awards.smithsonian_sarf_awards
GROUP BY start_year
ORDER BY start_year DESC;

In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 132.
SELECT
    priority,
    provenance,
    COUNT(*) AS cnt,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'smithsonian_sarf' AND priority = 132
GROUP BY priority, provenance;

## Handoff Notes

Contractor has no S3/Databricks access. Admin must upload the parquet, run this notebook, inspect the verification cells, and only then decide whether to wire any scheduled job. Do not add job YAML until the ingest has been run and verified.